# Диагностика открытия/закрытия гексагонов: work_mode vs заявки

## Назначение

Проверка предложения научного консультанта: можно ли определить фактическое открытие/закрытие гексагона

1. по переходам `work_mode`;
2. по устойчивому исчезновению/возникновению заявок;
3. насколько эти два определения совпадают.

## Важно

- Это **отдельный диагностический** ноутбук.
- Не меняет `main.tex`, DiD/event-study ноутбуки и production pipeline.
- Не пишет: «0 заявок = гексагон закрыт».
- Корректная формулировка: устойчивое отсутствие заявок после подтверждённой активности — эмпирический индикатор *возможного* закрытия.

## Данные и инфраструктура

- Загрузка только через `last_mile.io` (`application_dataset.csv`, `hexagons_dataset.csv`).
- Административный `change_type` — `last_mile.filter.classify_hexagons` (для сравнения, без переписывания).
- Логика диагностики: `scripts/hex_open_close_diagnostics_lib.py`.
- Полный batch-runner: `scripts/run_hex_open_close_diagnostics.py`.

## Выходы

- Таблицы: `outputs/hex_open_close/`
- Графики: `figures/hex_open_close/` (PDF + PNG через `last_mile.plot_style`)


In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SCRIPTS = PROJECT_ROOT / "scripts"
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

from last_mile.filter import (
    CLOSED,
    CORE_CHANGE_TYPES,
    INACTIVE_REGION,
    OPENED,
    STUDY_END,
    STUDY_START,
)
from last_mile.io import load_applications, load_hexagons
from last_mile.plot_style import PALETTE, save_figure, style_axes, with_plot_style
from hex_open_close_diagnostics_lib import (
    EVENT_PAD_DAYS,
    HORIZONS,
    PRIMARY_HORIZON,
    MIN_PRE_APPS,
    MIN_PRE_ACTIVE_DAYS,
    activity_change_timing,
    agreement_rate,
    build_daily_apps,
    build_event_panel,
    build_summary_table,
    change_type_vs_structural,
    classify_empirical,
    classify_permanent_and_temporary,
    compute_window_metrics,
    event_time_aggregates,
    lifetime_activity_diagnostics,
    low_demand_false_closure_risk,
    prepare_hex_universe,
    project_paths,
    structural_vs_empirical_table,
    suspicious_hexes,
    threshold_sensitivity_summary,
    work_mode_semantics,
)

paths = project_paths(PROJECT_ROOT)
OUT_DIR, FIG_DIR = paths["out"], paths["fig"]
print("STUDY_START/END:", STUDY_START.date(), STUDY_END.date())
print("INACTIVE_REGION sentinel:", INACTIVE_REGION)
print("OUT_DIR:", OUT_DIR.relative_to(PROJECT_ROOT).as_posix())
print("FIG_DIR:", FIG_DIR.relative_to(PROJECT_ROOT).as_posix())
print("PRIMARY_HORIZON:", PRIMARY_HORIZON, "HORIZONS:", HORIZONS)
print("MIN_PRE_APPS / MIN_PRE_ACTIVE_DAYS:", MIN_PRE_APPS, MIN_PRE_ACTIVE_DAYS)



STUDY_START/END: 2022-04-01 2022-10-18
INACTIVE_REGION sentinel: -100
OUT_DIR: outputs/hex_open_close
FIG_DIR: figures/hex_open_close
PRIMARY_HORIZON: 14 HORIZONS: (7, 14, 21, 28)
MIN_PRE_APPS / MIN_PRE_ACTIVE_DAYS: 5 3


## 0. Где сейчас задаётся `change_type` (не меняем)

Функция `classify_hexagons` в `last_mile/filter.py`:

- Активность зоны определяется **сентинелом** `region_id == -100` (`INACTIVE_REGION`), **не** `work_mode == 0`.
- `opened`: был неактивен (`region_id_old == -100`) → стал активен.
- `closed`: был активен → стал неактивен (`region_id_new == -100`).
- `work_mode` используется только чтобы отличить `workmode_only` / `region_and_workmode` при активных old/new регионах.

Правило «нет заявок ⇒ closed» **не** является правилом assignment. Отдельная эмпирическая диагностика уже есть в `last_mile/hex_activity.py` и `notebooks/hexagonsclosingcheck.ipynb` (admin `closed` vs заявки). Этот ноутбук отвечает на другой вопрос: transitions `work_mode` vs заявки.

Порог `min_orders_per_hex=5` в `build_analysis_panel` / `prepare_analysis_hex_universe` применяется **после** attach metadata и **до** DiD-панели. Здесь основной расчёт идёт **без** этого порога; параллельно показываем counts после `>=5`.


In [2]:
import inspect
from last_mile import filter as filter_mod

src = inspect.getsource(filter_mod.classify_hexagons)
display(Markdown("```python\n" + src + "\n```"))



```python
def classify_hexagons(hexagons: pd.DataFrame) -> pd.DataFrame:
    """
    Классифицирует каждый гексагон по типу воздействия и когорте.

    Активность зоны определяется сентинелом region_id == INACTIVE_REGION (-100),
    а не NaN: в данных region_id всегда заполнен, неактивная зона кодируется -100.

    Логика change_type:
      - был неактивен и остался неактивен        -> 'never_active'
      - был неактивен, стал активен              -> 'opened'
      - был активен, стал неактивен              -> 'closed'
      - активен, изменился только region_id      -> 'region_only'
      - активен, изменился только work_mode      -> 'workmode_only'
      - активен, изменились и регион, и режим     -> 'region_and_workmode'
      - прочие случаи                            -> 'other'

    NaT в treatment_date -> контрольная группа (never-treated).
    """
    df = hexagons.copy()

    active_old = df["region_id_old"] != INACTIVE_REGION
    active_new = df["region_id_new"] != INACTIVE_REGION
    same_region = df["region_id_old"] == df["region_id_new"]
    wm_changed = df["work_mode_old"] != df["work_mode_new"]

    df["change_type"] = OTHER
    df.loc[(~active_old) & (~active_new), "change_type"] = NEVER_ACTIVE
    df.loc[(~active_old) & active_new, "change_type"] = OPENED
    df.loc[active_old & (~active_new), "change_type"] = CLOSED
    df.loc[active_old & active_new & (~same_region) & (~wm_changed), "change_type"] = REGION_CHANGE
    df.loc[active_old & active_new & same_region & wm_changed, "change_type"] = WORKMODE_CHANGE
    df.loc[active_old & active_new & (~same_region) & wm_changed, "change_type"] = (
        REGION_AND_WORKMODE_CHANGE
    )

    df["cohort"] = df["treatment_date"]
    df["is_treated"] = df["treatment_date"].notna()

    return df

```

## 1. Семантика `work_mode`

Ничего не предполагаем. Смотрим уникальные значения, частоты, min/max, NaN, примеры и пересечение с `region_id == -100`.


In [3]:
applications = load_applications()
hexagons = load_hexagons()
print(
    f"applications={len(applications):,}, "
    f"hexagon_rows={len(hexagons):,}, "
    f"unique_hex={hexagons['hex'].nunique():,}"
)

sem = work_mode_semantics(hexagons)
display(sem["summaries"])
display(sem["value_counts"].sort_values(["field", "n"], ascending=[True, False]))
display(sem["examples"])
display(Markdown("### Cross: work_mode==0 vs region_id==-100"))
display(sem["wm_vs_region_cross"].head(15))

wm0_new = hexagons["work_mode_new"] == 0
reg_inact_new = hexagons["region_id_new"] == INACTIVE_REGION
print("P(region_new inactive | work_mode_new==0) =", float(reg_inact_new[wm0_new].mean()))
print("P(work_mode_new==0 | region_new inactive) =", float(wm0_new[reg_inact_new].mean()))



applications=1,807,426, hexagon_rows=3,644,904, unique_hex=3,629,072


,field,n,n_nan,n_unique,min,max,dtype,unique_values
0,work_mode_old,3644904,0,8,0.0,7.0,int64,"[0, 1, 2, 3, 4, 5, 6, 7]"
1,work_mode_new,3644904,0,8,0.0,7.0,int64,"[0, 1, 2, 3, 4, 5, 6, 7]"


,value,n,field,share
8,0,2971016,work_mode_new,0.815115
9,1,259239,work_mode_new,0.071124
10,7,176461,work_mode_new,0.048413
11,2,78740,work_mode_new,0.021603
12,3,68984,work_mode_new,0.018926
13,4,42768,work_mode_new,0.011734
14,5,28519,work_mode_new,0.007824
15,6,19177,work_mode_new,0.005261
0,0,2918381,work_mode_old,0.800674
1,7,334013,work_mode_old,0.091638


,hex,treatment_date,subject,region_id_old,region_id_new,work_mode_old,work_mode_new,example_group
0,88212662adfffff,2022-09-22,Оренбургская область,342,-100,7,0,wm_to_zero
1,8821233569fffff,2022-09-22,Оренбургская область,149,-100,2,0,wm_to_zero
2,882124d31bfffff,2022-09-22,Оренбургская область,349,-100,1,0,wm_to_zero
3,882123a89dfffff,2022-09-22,Оренбургская область,149,-100,2,0,wm_to_zero
4,8821244561fffff,2022-09-22,Оренбургская область,76,-100,7,0,wm_to_zero
5,88108e4249fffff,2022-09-22,Оренбургская область,-100,388,0,7,wm_from_zero
6,8821232729fffff,2022-09-22,Оренбургская область,-100,3482,0,1,wm_from_zero
7,8821231723fffff,2022-09-22,Оренбургская область,-100,9204,0,1,wm_from_zero
8,882104b435fffff,2022-09-22,Оренбургская область,-100,8761,0,7,wm_from_zero
9,882104b4e3fffff,2022-09-22,Оренбургская область,-100,8761,0,7,wm_from_zero


### Cross: work_mode==0 vs region_id==-100

,wm_old_zero,wm_new_zero,region_old_inactive,region_new_inactive,n
7,True,True,True,True,2598549
0,False,False,False,False,372075
2,False,True,False,True,351535
4,True,False,True,False,293467
5,True,True,False,True,10831
3,True,False,False,False,8346
6,True,True,True,False,7188
1,False,True,False,False,2913


P(region_new inactive | work_mode_new==0) = 0.9966001529443127
P(work_mode_new==0 | region_new inactive) = 1.0


### Structural classification по `work_mode`

| код | правило |
|---|---|
| A `explicit_closed` | `work_mode_old > 0` AND `work_mode_new == 0` |
| B `explicit_opened` | `work_mode_old == 0` AND `work_mode_new > 0` |
| C `remained_closed` | оба нуля |
| D `remained_active` | оба > 0 |

NaN / значения вне `{0..7}` — отдельно, без подмены нулями.


In [4]:
print(
    "NOTE: DiD universe applies min_orders_per_hex=5 AFTER cohort attach "
    "(last_mile.filter.build_analysis_panel / last_mile.hex_activity.prepare_analysis_hex_universe). "
    "Diagnostic default below = NO min-orders filter."
)

prep = prepare_hex_universe(
    applications, hexagons, min_orders_per_hex=None, apps_only=True
)
treated = prep["treated_meta"]
treated_diag = treated[~treated["excluded_cohort"]].copy()
print(
    "treated (app-linked):",
    len(treated),
    "| after excluding late cohort 2022-10-19:",
    len(treated_diag),
)

struct = (
    treated_diag["structural_wm"]
    .value_counts(dropna=False)
    .rename("n")
    .rename_axis("structural_wm")
    .reset_index()
)
struct["share"] = struct["n"] / struct["n"].sum()
display(struct)

prep5 = prepare_hex_universe(
    applications, hexagons, min_orders_per_hex=5, apps_only=True
)
t5 = prep5["treated_meta"]
t5 = t5[~t5["excluded_cohort"]]
struct5 = (
    t5["structural_wm"]
    .value_counts(dropna=False)
    .rename("n")
    .rename_axis("structural_wm")
    .reset_index()
)
struct5["share"] = struct5["n"] / struct5["n"].sum()
display(Markdown("### После DiD-фильтра min_orders >= 5"))
display(struct5)



NOTE: DiD universe applies min_orders_per_hex=5 AFTER cohort attach (last_mile.filter.build_analysis_panel / last_mile.hex_activity.prepare_analysis_hex_universe). Diagnostic default below = NO min-orders filter.
treated (app-linked): 28178 | after excluding late cohort 2022-10-19: 14832


,structural_wm,n,share
0,remained_active,11961,0.806432
1,explicit_closed,1370,0.092368
2,explicit_opened,893,0.060208
3,remained_closed,608,0.040992


### После DiD-фильтра min_orders >= 5

,structural_wm,n,share
0,remained_active,7970,0.854508
1,explicit_closed,765,0.082020
2,explicit_opened,363,0.038919
3,remained_closed,229,0.024552


## 2–4. Панель hex×day вокруг treatment и empirical open/close

Для treated hex (app-linked, без late cohort) строим календарь `[-42, +42]` относительно `treatment_date`, обрезанный реальным диапазоном заявок в study window. Отсутствующие hex-day заполняются нулями.

Один нулевой день **не** считается закрытием. Empirical правила (основной горизонт = 14):

- `empirical_closed_W`: pre-окно с подтверждённой активностью (`>=5` заявок и `>=3` active days) **и** W подряд дней post с `n_apps==0`.
- `empirical_opened_W`: W подряд дней pre с нулями **и** post с подтверждённой активностью.
- Если pre/post окно обрезано границей данных → `censored`, без классификации.


In [5]:
daily = build_daily_apps(prep["apps_study"])
panel = build_event_panel(
    treated_diag,
    daily,
    app_date_min=max(prep["app_date_min"], STUDY_START),
    app_date_max=min(prep["app_date_max"], STUDY_END),
    pad_days=EVENT_PAD_DAYS,
)
print(
    f"panel rows={len(panel):,}, zero_days={(panel['n_apps']==0).sum():,}, "
    f"sum n_apps={int(panel['n_apps'].sum()):,}"
)
assert panel.duplicated(["hex", "date"]).sum() == 0

metrics = compute_window_metrics(panel, horizons=HORIZONS)
for h in HORIZONS:
    metrics = classify_empirical(metrics, horizon=h)

display(Markdown("### Pre/post metrics snapshot (primary W=14)"))
cols14 = [
    "hex",
    "structural_wm",
    "change_type",
    "full_pre_14",
    "full_post_14",
    "left_censored",
    "right_censored",
    "pre_apps_14",
    "post_apps_14",
    "pre_active_days_14",
    "post_active_days_14",
    "pre_max_consecutive_zero_14",
    "post_max_consecutive_zero_14",
    "empirical_14",
]
display(metrics[cols14].head(10))

sens = threshold_sensitivity_summary(metrics, HORIZONS)
display(Markdown("### Sensitivity 7/14/21/28"))
display(sens)



panel rows=1,137,792, zero_days=1,052,820, sum n_apps=285,828


### Pre/post metrics snapshot (primary W=14)

,hex,structural_wm,change_type,full_pre_14,full_post_14,left_censored,right_censored,pre_apps_14,post_apps_14,pre_active_days_14,post_active_days_14,pre_max_consecutive_zero_14,post_max_consecutive_zero_14,empirical_14
0,880b3086a1fffff,explicit_opened,opened,True,True,False,False,0,0,0,0,14,14,unclear
1,880b3086a9fffff,explicit_opened,opened,True,True,False,False,16,3,8,2,3,10,remained_active
2,880b3086abfffff,explicit_opened,opened,True,True,False,False,0,4,0,1,14,10,unclear
3,880b3086e7fffff,explicit_opened,opened,True,True,False,False,2,0,1,0,13,14,unclear
4,880b30adadfffff,remained_closed,never_active,True,True,False,False,0,1,0,1,14,8,unclear
5,880b34040bfffff,remained_active,other,True,True,False,False,0,0,0,0,14,14,unclear
6,880b340491fffff,remained_active,other,True,True,False,False,0,0,0,0,14,14,unclear
7,880b34049bfffff,remained_active,other,True,True,False,False,1,7,1,2,10,5,unclear
8,880b3404c1fffff,remained_active,other,True,True,False,False,0,0,0,0,14,14,unclear
9,880b3404c9fffff,remained_active,other,True,True,False,False,0,0,0,0,14,14,unclear


### Sensitivity 7/14/21/28

,horizon,n_explicit_closed,n_explicit_opened,n_explicit_closed_empirical_closed,share_closed_confirmed,n_explicit_opened_empirical_opened,share_opened_confirmed,n_hidden_empirical_closed_among_remained_active,n_explicit_closed_but_post_apps,n_empirical_closed,n_empirical_opened,n_unclear,n_censored,n_remained_active_emp,n_comparable_vs_primary14,n_status_changes_vs_primary14,share_status_changes_vs_primary14
0,7,1370,893,1,0.000730,1,0.001120,53,286,56,34,13853,0,889,14832,832,0.056095
1,14,1370,893,8,0.005839,3,0.003359,94,389,104,79,13103,0,1546,14832,0,0.000000
2,21,1370,893,4,0.002920,3,0.003359,111,514,121,124,12529,0,2058,14832,695,0.046858
3,28,1370,893,4,0.002920,2,0.002240,55,581,65,71,5791,7683,1222,7149,586,0.081970


## 5. Permanent closure vs temporary shutdown


In [6]:
perm_tmp = classify_permanent_and_temporary(
    panel, metrics, horizon=PRIMARY_HORIZON
)
display(perm_tmp["permanent_status"].value_counts())
print("temporary_shutdown:", int(perm_tmp["temporary_shutdown"].sum()))
print(
    "empirical_permanent_closure:",
    int(perm_tmp["empirical_permanent_closure"].sum()),
)

display(Markdown("### Распределение longest post zero-run"))
display(
    perm_tmp["longest_post_zero_run"].describe(
        percentiles=[0.5, 0.75, 0.9, 0.95]
    )
)



permanent_status
right_censored_cannot_claim                    7683
not_permanent                                  7135
empirical_permanent_closure_after_spillover       9
empirical_permanent_closure                       5
Name: count, dtype: int64

temporary_shutdown: 32
empirical_permanent_closure: 14


### Распределение longest post zero-run

count    14832.000000
mean        25.948557
std         11.982045
min          0.000000
50%         27.000000
75%         35.000000
90%         43.000000
95%         43.000000
max         43.000000
Name: longest_post_zero_run, dtype: float64

## 6. Сопоставление structural work_mode × empirical activity


In [7]:
for h in HORIZONS:
    ct = structural_vs_empirical_table(metrics, horizon=h)
    display(Markdown(f"### Horizon={h}"))
    pivot = ct.pivot_table(
        index="structural_wm", columns="empirical", values="n", fill_value=0
    )
    display(pivot)
    print(agreement_rate(metrics, h))

display(Markdown("### Ключевые расхождения (W=14)"))
m = metrics
q1 = m[m["structural_wm"] == "explicit_closed"]
q2 = m[m["structural_wm"] == "explicit_opened"]
q3 = m[
    (m["structural_wm"] == "remained_active")
    & (m["empirical_14"] == "empirical_closed")
]
q4 = m[(m["work_mode_new"] == 0) & (m["post_apps_14"] > 0)]
print(
    f"1) explicit_closed → emp_closed_14: "
    f"{(q1['empirical_14']=='empirical_closed').sum()}/{len(q1)}"
)
print(f"   с достаточным pre-activity: {(q1['pre_sufficient_activity_14']).sum()}")
print(f"   post_apps_14>0: {(q1['post_apps_14']>0).sum()}")
print(
    f"2) explicit_opened → emp_opened_14: "
    f"{(q2['empirical_14']=='empirical_opened').sum()}/{len(q2)}"
)
print(f"3) hidden empirical closed among remained_active: {len(q3)}")
print(f"4) work_mode_new==0 but post_apps_14>0: {len(q4)}")

sus = suspicious_hexes(metrics, PRIMARY_HORIZON)
display(Markdown("### Примеры suspicious hex"))
preview_cols = [
    c for c in ("hex", "treatment_date", "structural_wm", "change_type", "empirical_14", "post_apps_14")
    if c in sus.columns
]
print(f"suspicious_hexes: n={len(sus)}; showing head(5) of key columns")
display(sus.loc[:, preview_cols].head(5))



### Horizon=7

empirical,empirical_closed,empirical_opened,remained_active,unclear
structural_wm,,,,
explicit_closed,1.0,4.0,89.0,1276.0
explicit_opened,1.0,1.0,1.0,890.0
remained_active,53.0,29.0,799.0,11080.0
remained_closed,1.0,0.0,0.0,607.0


{'horizon': 7, 'n_explicit_closed': 1370, 'n_explicit_opened': 893, 'n_explicit_closed_empirical_closed': 1, 'share_closed_confirmed': 0.00072992700729927, 'n_explicit_opened_empirical_opened': 1, 'share_opened_confirmed': 0.0011198208286674132, 'n_hidden_empirical_closed_among_remained_active': 53, 'n_explicit_closed_but_post_apps': 286, 'n_empirical_closed': 56, 'n_empirical_opened': 34}


### Horizon=14

empirical,empirical_closed,empirical_opened,remained_active,unclear
structural_wm,,,,
explicit_closed,8.0,4.0,153.0,1205.0
explicit_opened,1.0,3.0,7.0,882.0
remained_active,94.0,72.0,1386.0,10409.0
remained_closed,1.0,0.0,0.0,607.0


...

### Ключевые расхождения (W=14)

1) explicit_closed → emp_closed_14: 8/1370
   с достаточным pre-activity: 161
   post_apps_14>0: 389
2) explicit_opened → emp_opened_14: 3/893
3) hidden empirical closed among remained_active: 94
4) work_mode_new==0 but post_apps_14>0: 497


### Примеры suspicious hex

suspicious_hexes: n=1116; showing head(5) of key columns


                 hex treatment_date    structural_wm change_type  \
0  8810881001fffff     2022-09-22  explicit_closed      closed   
1  881088132bfffff     2022-09-22  explicit_closed      closed   
2  88108a69adfffff     2022-09-22  explicit_closed      closed   
3  88108a6b5bfffff     2022-09-22  explicit_closed      closed   
4  88108a834bfffff     2022-09-22  explicit_closed      closed   

      empirical_14  post_apps_14  
0  empirical_closed             0  
1  empirical_closed             0  
2  empirical_closed             0  
3  empirical_closed             0  
4  empirical_closed             0  


## 7. Тайминг фактического изменения активности vs `treatment_date`


In [8]:
timing = activity_change_timing(metrics)
closed_t = timing.loc[
    timing["structural_wm"] == "explicit_closed", "closure_timing_rel"
].dropna()
opened_t = timing.loc[
    timing["structural_wm"] == "explicit_opened", "opening_timing_rel"
].dropna()
print("closure timing (last active - treatment) describe:")
print(closed_t.describe())
print("opening timing (first post active - treatment) describe:")
print(opened_t.describe())



closure timing (last active - treatment) describe:
count    893.000000
mean      -6.599104
std       18.011907
min      -42.000000
25%      -18.000000
50%       -5.000000
75%       -1.000000
max       42.000000
Name: closure_timing_rel, dtype: float64
opening timing (first post active - treatment) describe:
count    324.000000
mean      16.947531
std       12.221144
min        0.000000
25%        7.000000
50%       14.500000
75%       26.000000
max       42.000000
Name: opening_timing_rel, dtype: float64


## 8. Графики

Единый стиль проекта (`last_mile.plot_style`). Runner уже пишет полный набор в `figures/hex_open_close/`; ниже пересобираются ключевые event-time charts.


In [9]:
existing = sorted(FIG_DIR.glob("*.pdf"))
print("Existing figures:", [p.name for p in existing])

event_agg = event_time_aggregates(panel)
GROUP_COLORS = {
    "explicit_closed": "#B85C38",
    "explicit_opened": "#4C78A8",
    "remained_active": "#234E70",
}


@with_plot_style
def _plot_event(ycol, ylab, fname):
    fig, ax = plt.subplots(figsize=(8.2, 4.4))
    for gname, g in event_agg.groupby("structural_wm"):
        ax.plot(
            g["rel_day"],
            g[ycol],
            label=gname,
            color=GROUP_COLORS.get(gname, PALETTE["text"]),
            linewidth=1.6,
        )
    ax.axvline(0, color=PALETTE["zero"], linewidth=0.9, linestyle=":")
    ax.set_xlabel("День относительно treatment_date")
    ax.set_ylabel(ylab)
    ax.legend(loc="best", frameon=False)
    style_axes(ax)
    save_figure(fig, FIG_DIR / fname, preview_dpi=300)


_plot_event(
    "mean_n_apps",
    "Среднее число заявок на hex-day",
    "applications_event_time.pdf",
)
_plot_event(
    "active_share",
    "Доля hex с n_apps > 0",
    "active_share_event_time.pdf",
)
print("Updated event-time figures.")



Existing figures: ['active_share_event_time.pdf', 'applications_event_time.pdf', 'applications_event_time_closed.pdf', 'applications_event_time_opened.pdf', 'closure_heatmap.pdf', 'closure_timing_hist.pdf', 'opening_heatmap.pdf', 'opening_timing_hist.pdf']
Updated event-time figures.


## 9. Связь с текущим `change_type`


In [10]:
ct_vs = change_type_vs_structural(metrics)
pivot = ct_vs.pivot_table(
    index="change_type", columns="structural_wm", values="n", fill_value=0
)
display(pivot)

display(
    Markdown(
        '''
**Текущие правила opened/closed (`classify_hexagons`):** только `region_id` ↔ `-100`.

**Наблюдение:** большинство `explicit_closed` по work_mode совпадают с admin `closed`, но не все
(часть попадает в `workmode_only`, где регион остаётся активным, а `work_mode` падает до 0).
Это критично: `work_mode_new==0` **не эквивалентно** administratively closed.
'''
    )
)



structural_wm,explicit_closed,explicit_opened,remained_active,remained_closed
change_type,,,,
closed,851.0,0.0,0.0,25.0
never_active,0.0,0.0,0.0,569.0
opened,0.0,854.0,0.0,14.0
other,0.0,0.0,5213.0,0.0
region_and_workmode,37.0,23.0,3771.0,0.0
region_only,0.0,0.0,826.0,0.0
workmode_only,482.0,16.0,2151.0,0.0



**Текущие правила opened/closed (`classify_hexagons`):** только `region_id` ↔ `-100`.

**Наблюдение:** большинство `explicit_closed` по work_mode совпадают с admin `closed`, но не все
(часть попадает в `workmode_only`, где регион остаётся активным, а `work_mode` падает до 0).
Это критично: `work_mode_new==0` **не эквивалентно** administratively closed.


## 10–11. Lifetime activity и low-demand false-closure risk


In [11]:
life = lifetime_activity_diagnostics(
    prep["apps_study"],
    treated_diag[
        [
            "hex",
            "treatment_date",
            "structural_wm",
            "change_type",
            "is_treated",
            "n_orders_study",
        ]
    ],
)
display(life["lifetime_pattern"].value_counts())

low = low_demand_false_closure_risk(metrics, perm_tmp, HORIZONS)
display(Markdown("### P(zero-run ≥ W | later resumed), by pre-volume"))
display(low)



lifetime_pattern
intermittent_or_low_demand       6463
closed_like                      6353
temporarily_inactive_or_mixed    1027
low_demand                        703
opened_like                       284
active_throughout                   1
never_active                        1
Name: count, dtype: int64

### P(zero-run ≥ W | later resumed), by pre-volume

,horizon,pre_volume_bin,n_hex_resumed,n_with_zero_run_ge_w,share_with_zero_run_ge_w,n_temporary_shutdown_flag
0,7,1-4,4196,4176,0.995234,0
1,7,5-9,949,906,0.954689,12
2,7,10-24,973,799,0.821172,18
3,7,25-49,450,140,0.311111,0
4,7,50+,513,8,0.015595,0
5,14,1-4,4196,3663,0.872974,0
6,14,5-9,949,587,0.618546,12
7,14,10-24,973,296,0.304214,18
8,14,25-49,450,12,0.026667,0
9,14,50+,513,0,0.000000,0


## 12. Сохранение outputs

Полный набор также пишет `scripts/run_hex_open_close_diagnostics.py`. Ниже — ключевые таблицы текущего прогона.


In [12]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

struct.to_csv(OUT_DIR / "structural_wm_counts.csv", index=False)
sens.to_csv(OUT_DIR / "closure_threshold_sensitivity.csv", index=False)
cross = pd.concat(
    [
        structural_vs_empirical_table(metrics, horizon=h).assign(horizon=h)
        for h in HORIZONS
    ],
    ignore_index=True,
)
cross.to_csv(OUT_DIR / "structural_vs_empirical.csv", index=False)
perm_tmp.to_csv(OUT_DIR / "zero_run_diagnostics.csv", index=False)
timing.to_csv(OUT_DIR / "activity_change_timing.csv", index=False)
sus.to_csv(OUT_DIR / "suspicious_hexes.csv", index=False)
metrics.to_csv(OUT_DIR / "hex_open_close_metrics.csv", index=False)
life.to_csv(OUT_DIR / "lifetime_activity_diagnostics.csv", index=False)
low.to_csv(OUT_DIR / "low_demand_false_closure_risk.csv", index=False)

summary = build_summary_table(
    struct,
    sens,
    perm_tmp,
    n_treated=len(treated_diag),
    min_orders_note="none (diagnostic); DiD uses >=5 after attach",
)
summary["n_admin_closed_change_type"] = int((metrics["change_type"] == CLOSED).sum())
summary["n_admin_opened_change_type"] = int((metrics["change_type"] == OPENED).sum())
summary["n_suspicious_rows"] = int(len(sus))
summary.to_csv(OUT_DIR / "hex_open_close_summary.csv", index=False)
display(summary.T)
print("Saved to", OUT_DIR.relative_to(PROJECT_ROOT).as_posix())



,0
n_treated_hex_diagnosed,14832
min_orders_filter,none (diagnostic); DiD uses >=5 after attach
n_explicit_closed,1370
n_explicit_opened,893
n_empirical_closed_14,104
n_empirical_opened_14,79
share_closed_confirmed_14,0.005839
share_opened_confirmed_14,0.003359
n_temporary_shutdown,32
n_permanent_closure,14


Saved to outputs/hex_open_close


## Вывод

В данных наблюдаются и административные, и эмпирические переходы активности гексагонов, но они слабо совпадают. Структурные классы `explicit_closed` / `explicit_opened` строятся по смене `work_mode` в 0 / из 0, тогда как active/inactive в основном assignment (`classify_hexagons`) определяется сентинелом `region_id == -100`, а не `work_mode`. Эмпирическое состояние определяется по заявкам: устойчивое отсутствие (или появление) заявок после подтверждённой pre-активности на горизонте \(W\in\{7,14,21,28\}\) (основной \(W=14\)).

Для \(W=14\) из 1370 `explicit_closed` только 8 подтверждаются как `empirical_closed` (доля ≈0.006), а из 893 `explicit_opened` — 3 как `empirical_opened`. Событийные профили `applications` и `active_share` вокруг даты изменения используются как описательная иллюстрация динамики, а не как отдельный causal estimand. Скрытые эмпирические закрытия среди `remained_active` малочисленны относительно выборки; при этом низкий спрос сам по себе часто порождает длинные нулевые окна, поэтому «0 заявок» не тождественно закрытию.

Следовательно, открытие/закрытие по заявкам — диагностическая robustness-проверка к угрозе selective attrition/composition вокруг даты изменения, а не новый основной DiD-estimand. По результатам ноутбука нет оснований считать, что систематическое расхождение `work_mode`-статуса и заявковой активности разрушает базовую интерпретацию основной DiD-панели; сохраняется лишь предостережение о ложноположительных «закрытиях» при низкой интенсивности спроса.


In [13]:
n_ec = int((metrics["structural_wm"] == "explicit_closed").sum())
n_eo = int((metrics["structural_wm"] == "explicit_opened").sum())
n_emp_c = int((metrics["empirical_14"] == "empirical_closed").sum())
n_emp_o = int((metrics["empirical_14"] == "empirical_opened").sum())
agr = agreement_rate(metrics, 14)
n_tmp = int(perm_tmp["temporary_shutdown"].sum())
n_perm = int(perm_tmp["empirical_permanent_closure"].sum())
n_hidden = int(
    (
        (metrics["structural_wm"] == "remained_active")
        & (metrics["empirical_14"] == "empirical_closed")
    ).sum()
)
n_wm0_apps = int(((metrics["work_mode_new"] == 0) & (metrics["post_apps_14"] > 0)).sum())
n_chg7 = int(sens.loc[sens.horizon == 7, "n_status_changes_vs_primary14"].iloc[0])
n_chg21 = int(sens.loc[sens.horizon == 21, "n_status_changes_vs_primary14"].iloc[0])
n_chg28 = int(sens.loc[sens.horizon == 28, "n_status_changes_vs_primary14"].iloc[0])

suff = metrics[
    (metrics.structural_wm == "explicit_closed") & (metrics.pre_sufficient_activity_14)
]
n_suff = len(suff)
n_suff_closed = int((suff.empirical_14 == "empirical_closed").sum())

risk_14_5_9 = low[(low.horizon == 14) & (low.pre_volume_bin == "5-9")]
risk_share = (
    float(risk_14_5_9["share_with_zero_run_ge_w"].iloc[0])
    if len(risk_14_5_9)
    else float("nan")
)

lines = []
lines.append("### 1. Можно ли надёжно считать `work_mode_new == 0` закрытием?")
lines.append("")
lines.append(
    "**Нет, не само по себе.** В данных `work_mode ∈ {0..7}`, NaN нет. "
    "Значение `0` сильно коррелирует с `region_id_new == -100`, но не тождественно ему. "
    "Среди app-linked treated есть `explicit_closed` в `workmode_only` "
    "(регион остаётся активным). Административный `closed` в pipeline определяется "
    "**только** через `region_id == -100` (`classify_hexagons`)."
)
lines.append("")
lines.append("### 2. Соответствует ли это фактическому исчезновению заявок?")
lines.append("")
lines.append(
    f"**Слабо / выборочно.** Из **{n_ec}** `explicit_closed` только "
    f"**{agr['n_explicit_closed_empirical_closed']}** подтверждаются как "
    f"`empirical_closed_14` (доля ≈ {agr['share_closed_confirmed']:.3f})."
)
lines.append(
    "При этом у большинства `explicit_closed` post-окно 14 дней может быть нулевым, "
    "но критерий достаточной pre-активности не выполняется (low-demand)."
)
lines.append(
    f"Среди {n_suff} `explicit_closed` с достаточной pre-активностью только "
    f"**{n_suff_closed}** действительно имеют 14 нулевых post-дней; "
    f"у остальных заявки продолжаются."
)
lines.append(
    f"Случаев `work_mode_new==0` при `post_apps_14>0`: **{n_wm0_apps}**."
)
lines.append("")
lines.append("### 3. Можно ли обнаруживать дополнительные закрытия только по applications?")
lines.append("")
lines.append(
    f"**Да, но только как диагностику, не как assignment.** "
    f"Независимый индикатор находит **{n_emp_c}** `empirical_closed_14` и "
    f"**{n_emp_o}** `empirical_opened_14`; среди них **{n_hidden}** — "
    "скрытые среди structural `remained_active`."
)
lines.append("")
lines.append("### 4. Какой минимальный zero-run разумно использовать: 7 / 14 / 21 / 28?")
lines.append("")
lines.append("- **7 дней** слишком агрессивен для low-demand hex.")
lines.append(
    f"- **14 дней** — разумный основной диагностический горизонт "
    f"(bin 5–9 share≈{risk_share:.2f}; 25+ почти 0)."
)
lines.append("- **21–28** строже и сильнее упираются в right-censoring.")
lines.append("")
lines.append("### 5. Сколько hex меняют классификацию от порога?")
lines.append("")
lines.append(
    f"Vs primary 14: на 7d ~{n_chg7}; на 21d ~{n_chg21}; на 28d ~{n_chg28}. "
    "Нужен sensitivity table."
)
lines.append("")
lines.append("### 6. Есть ли существенное число временных пауз?")
lines.append("")
lines.append(
    f"temporary_shutdown = **{n_tmp}**; permanent_closure = **{n_perm}**. "
    "Lifetime чаще intermittent/low-demand или closed-like."
)
lines.append("")
lines.append("### 7. Совпадает ли фактическая дата изменения активности с `treatment_date`?")
lines.append("")
lines.append(
    "См. `closure_timing_hist` / `opening_timing_hist` и "
    "`activity_change_timing.csv`. Часть hex имеет лаг/лид относительно t=0."
)
lines.append("")
lines.append("### 8. Нужно ли менять `change_type` / выборку основного DiD?")
lines.append("")
lines.append(
    "**Пока нет.** Admin closed/opened должны оставаться administrative "
    "(`region_id==-100`). `work_mode==0` нельзя молча трактовать как закрытие. "
    "Эмпирика по заявкам — только diagnostics/robustness."
)
lines.append("")
lines.append("Если позже менять pipeline, кандидаты (**сейчас не трогаем**):")
lines.append("")
lines.append("1. `last_mile/filter.py` — `classify_hexagons`.")
lines.append(
    "2. `last_mile/hex_activity.py` / `notebooks/hexagonsclosingcheck.ipynb`."
)
lines.append(
    "3. DiD sample builders (`build_analysis_panel`, основные notebooks) — "
    "только при отдельном методологическом решении."
)
display(Markdown("\n".join(lines)))



### 1. Можно ли надёжно считать `work_mode_new == 0` закрытием?

**Нет, не само по себе.** В данных `work_mode ∈ {0..7}`, NaN нет. Значение `0` сильно коррелирует с `region_id_new == -100`, но не тождественно ему. Среди app-linked treated есть `explicit_closed` в `workmode_only` (регион остаётся активным). Административный `closed` в pipeline определяется **только** через `region_id == -100` (`classify_hexagons`).

### 2. Соответствует ли это фактическому исчезновению заявок?

**Слабо / выборочно.** Из **1370** `explicit_closed` только **8** подтверждаются как `empirical_closed_14` (доля ≈ 0.006).
При этом у большинства `explicit_closed` post-окно 14 дней может быть нулевым, но критерий достаточной pre-активности не выполняется (low-demand).
Среди 161 `explicit_closed` с достаточной pre-активностью только **8** действительно имеют 14 нулевых post-дней; у остальных заявки продолжаются.
Случаев `work_mode_new==0` при `post_apps_14>0`: **497**.

### 3. Можно ли обнаруживать дополнительные закрытия только по applications?

**Да, но только как диагностику, не как assignment.** Независимый индикатор находит **104** `empirical_closed_14` и **79** `empirical_opened_14`; среди них **94** — скрытые среди structural `remained_active`.

### 4. Какой минимальный zero-run разумно использовать: 7 / 14 / 21 / 28?

- **7 дней** слишком агрессивен для low-demand hex.
- **14 дней** — разумный основной диагностический горизонт (bin 5–9 share≈0.62; 25+ почти 0).
- **21–28** строже и сильнее упираются в right-censoring.

### 5. Сколько hex меняют классификацию от порога?

Vs primary 14: на 7d ~832; на 21d ~695; на 28d ~586. Нужен sensitivity table.

### 6. Есть ли существенное число временных пауз?

temporary_shutdown = **32**; permanent_closure = **14**. Lifetime чаще intermittent/low-demand или closed-like.

### 7. Совпадает ли фактическая дата изменения активности с `treatment_date`?

См. `closure_timing_hist` / `opening_timing_hist` и `activity_change_timing.csv`. Часть hex имеет лаг/лид относительно t=0.

### 8. Нужно ли менять `change_type` / выборку основного DiD?

**Пока нет.** Admin closed/opened должны оставаться administrative (`region_id==-100`). `work_mode==0` нельзя молча трактовать как закрытие. Эмпирика по заявкам — только diagnostics/robustness.

Если позже менять pipeline, кандидаты (**сейчас не трогаем**):

1. `last_mile/filter.py` — `classify_hexagons`.
2. `last_mile/hex_activity.py` / `notebooks/hexagonsclosingcheck.ipynb`.
3. DiD sample builders (`build_analysis_panel`, основные notebooks) — только при отдельном методологическом решении.